<a href="https://colab.research.google.com/github/shribr/Bricky/blob/main/Tools/torso-embeddings/Bricky-train-face-encoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bricky — Face Embedding Training

End-to-end pipeline that produces the on-device face identification model.
Run cells top-to-bottom. Total time on a free Colab T4: **~3-5 hours**.

The face crop covers rows **17–35%** of the figure image — below the
hairline, above the neck — capturing printed expressions, skin tone,
glasses, and facial hair while **excluding hair/helmets/hats**.

**What you need before starting:**
1. Switch the runtime to a GPU: `Runtime > Change runtime type > T4 GPU`.
2. Nothing else. The notebook clones the repo, downloads the dataset, trains, and packages the artifacts.

**Output:** at the end you get a `face_embeddings_bundle.zip` to download. Unzip into your local repo at `Bricky/Resources/FaceEmbeddings/`, run `xcodegen generate`, rebuild — done.

**Can run in parallel with the torso notebook** — they use separate data dirs and output dirs.

## 1. Verify GPU

In [ ]:
!nvidia-smi || echo 'NO GPU — switch runtime to T4 before continuing'

## 2. Install dependencies

Colab already has torch + torchvision + Pillow. We add `coremltools` for the final export.

In [ ]:
!pip install -q coremltools numpy pillow requests

## 3. Clone the repo

Public repo, no token needed. If re-running, the old clone is removed first.

In [ ]:
import shutil, os

if os.path.exists('Bricky'):
    shutil.rmtree('Bricky')

!git clone --depth 1 https://github.com/shribr/Bricky.git
%cd Bricky
!pwd

## 4. Build the training dataset (~30 min)

Downloads ~14K face reference renders from Rebrickable CDN. Each is cropped to the face band (17–35%) and saved as a 224×224 JPEG. Polite delay between requests so we don't hammer the CDN.

~1,600 figures will 404 (no image on Rebrickable) — that is normal.

In [ ]:
!python3 Tools/torso-embeddings/build-face-dataset.py --sleep 0.02

### Sanity check and manifest recovery

Count images and verify the manifest exists. If the build step was interrupted after downloading images but before writing the manifest, this cell regenerates it from whatever images are on disk.

In [ ]:
import glob, json, os
from PIL import Image
from IPython.display import display

data_dir = 'Tools/torso-embeddings/data-face'
figs_dir = os.path.join(data_dir, 'figures')
manifest_path = os.path.join(data_dir, 'manifest.json')

imgs = sorted(glob.glob(os.path.join(figs_dir, '*.jpg')))
print(f'Have {len(imgs)} face crops')

# Regenerate manifest if missing (build step was interrupted)
if not os.path.exists(manifest_path) and imgs:
    entries = [{"id": os.path.splitext(os.path.basename(f))[0], "name": ""} for f in imgs]
    with open(manifest_path, 'w') as f:
        json.dump({"figures": entries}, f, separators=(",", ":"))
    print(f'Regenerated manifest with {len(entries)} entries')
elif os.path.exists(manifest_path):
    m = json.load(open(manifest_path))
    print(f'Manifest has {len(m["figures"])} entries')
else:
    print('ERROR: No images found. Re-run the build step above.')

if imgs:
    display(Image.open(imgs[len(imgs) // 2]).resize((112, 112)))

## 5. Train the encoder (~2-4 hrs on T4)

Self-supervised contrastive (SimCLR-style). Two augmented views of each face crop are pulled together; views from different figures are pushed apart. Defaults: 20 epochs, batch 128, ResNet18 backbone.

If Colab disconnects before this finishes you can lower `--epochs 12` for a faster run with slightly lower accuracy.

In [ ]:
!python3 Tools/torso-embeddings/train-face-encoder.py \
    --epochs 20 \
    --batch-size 128 \
    --num-workers 2

## 6. Embed the full catalog (~10 min)

Runs the trained encoder over every face crop and writes the bundled vector index that ships with the iOS app. Explicit paths avoid any CWD confusion in Colab.

In [ ]:
!python3 Tools/torso-embeddings/embed-face-catalog.py \
    --checkpoint Tools/torso-embeddings/out-face/face_encoder.pt \
    --data Tools/torso-embeddings/data-face \
    --output Bricky/Resources/FaceEmbeddings

## 7. Export to CoreML (~1 min)

Converts the PyTorch encoder to CoreML for on-device inference.

> **Note:** Colab has a known `BlobWriter not loaded` bug with the `mlprogram` format in `coremltools`. We use `neuralnetwork` format instead, which works identically on iOS 17+ and avoids the bug.

In [ ]:
import sys, os
sys.path.insert(0, 'Tools/torso-embeddings')
from pathlib import Path
from importlib.machinery import SourceFileLoader

_TRAINER = SourceFileLoader(
    'train_face_encoder',
    'Tools/torso-embeddings/train-face-encoder.py'
).load_module()
FaceEncoder = _TRAINER.FaceEncoder

import torch
import torch.nn as nn
import coremltools as ct

class InferenceWrapper(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
    def forward(self, x):
        return self.encoder.encode(x)

ckpt_path = 'Tools/torso-embeddings/out-face/face_encoder.pt'
ckpt = torch.load(ckpt_path, map_location='cpu')
model = FaceEncoder(embed_dim=ckpt.get('embed_dim', 256))
model.load_state_dict(ckpt['state_dict'])
model.eval()
wrapped = InferenceWrapper(model).eval()

example = torch.randn(1, 3, 224, 224)
traced = torch.jit.trace(wrapped, example)

image_input = ct.ImageType(
    name='image',
    shape=(1, 3, 224, 224),
    scale=1.0 / 255.0 / 0.226,
    bias=[-0.485 / 0.229, -0.456 / 0.224, -0.406 / 0.225],
    color_layout=ct.colorlayout.RGB,
)

mlmodel = ct.convert(
    traced,
    inputs=[image_input],
    outputs=[ct.TensorType(name='embedding')],
    convert_to='neuralnetwork',
)
mlmodel.short_description = 'Bricky face-region encoder (ResNet18, 512-D, L2-normalized)'

out_path = Path('Bricky/Resources/FaceEmbeddings/FaceEncoder.mlmodel')
out_path.parent.mkdir(parents=True, exist_ok=True)
mlmodel.save(str(out_path))
print(f'Wrote {out_path} ({out_path.stat().st_size / (1024*1024):.1f} MB)')

## 8. Package and download

Bundles all three artifacts into `face_embeddings_bundle.zip`.

In [ ]:
import os, shutil
from pathlib import Path

out_dir = Path('Bricky/Resources/FaceEmbeddings')
expected = ['face_embeddings.bin', 'face_embeddings_index.json', 'FaceEncoder.mlmodel']
missing = [e for e in expected if not (out_dir / e).exists()]
if missing:
    raise RuntimeError(f'Pipeline did not produce: {missing}')

zip_path = '/content/face_embeddings_bundle'
shutil.make_archive(zip_path, 'zip', out_dir)
size_mb = os.path.getsize(zip_path + '.zip') / (1024 * 1024)
print(f'Wrote {zip_path}.zip ({size_mb:.1f} MB)')
for e in expected:
    p = out_dir / e
    print(f'  {e}: {p.stat().st_size / (1024 * 1024):.2f} MB')

In [ ]:
from google.colab import files
files.download('/content/face_embeddings_bundle.zip')

## 9. Install on your Mac

Run these commands locally after the download completes:

```bash
cd /path/to/Bricky
mkdir -p Bricky/Resources/FaceEmbeddings
unzip -o ~/Downloads/face_embeddings_bundle.zip -d Bricky/Resources/FaceEmbeddings/
xcodegen generate
xcodebuild -project Bricky.xcodeproj -scheme Bricky \
    -destination 'platform=iOS Simulator,id=YOUR_SIMULATOR_ID' \
    build
```

On the next scan, the cascade automatically uses the trained model. The runtime checks `FaceEmbeddingService.shared.isAvailable` and lights up the face embedding phase once the artifacts are present.